In [ ]:
import os
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import contextily as ctx

In [ ]:
os.chdir('..')

In [ ]:
from analysis.analysis_utils import plot_city_lot_map, plot_rose_diagram
from data_processing.lot_engineering import get_orientation

In [ ]:
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10
})

In [ ]:
LOTS_PATH = "data/lots/lot_features.geojson"
BOUNDARIES_PATH = "data/lots/city_boundaries.geojson"

In [ ]:
lots = gpd.read_file(LOTS_PATH)

In [ ]:
boundaries = gpd.read_file(BOUNDARIES_PATH)
boundaries.to_crs(epsg=5070, inplace=True)

In [ ]:
lots.head()

# Lot Maps

In [ ]:
# San Bernardino map
plot_city_lot_map(lots, boundaries, "san-bernardino-ca", "Parking lots in San Bernardino, California.")

In [ ]:
# Long Beach
plot_city_lot_map(lots, boundaries, "long-beach-ca", "Parking area is unevenly distributed among lots in Long Beach, California.", -0.01)

In [ ]:
# Houston
plot_city_lot_map(lots, boundaries, "houston-tx", "Parking in Houston, Texas, is evenly distributed and ordered within the city grid.", 0.02)

In [ ]:
# Austin
plot_city_lot_map(lots, boundaries, "san-antonio-tx", "Parking lots in San Antonio, Texas, have high orientation entropy.", -0.01)

# Lot Features

In [ ]:
plt.figure(figsize=(8, 6), dpi=200)

ax = sns.histplot(lots["pct_lot_area"], bins=25, kde=True, stat="percent")

sns.despine(ax=ax, top=True, right=True)

ax.set_title("Distribution of Lot Area %")
ax.set_xlabel("Lot Area %")
ax.set_ylabel("% of Cities")

ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.show()

In [ ]:
plt.figure(figsize=(8, 6), dpi=200)

ax = sns.histplot(lots["avg_lot_area"], bins=25, kde=True, stat="percent")

sns.despine(ax=ax, top=True, right=True)

ax.set_title("Distribution of Average Lot Size")
ax.set_xlabel("Average Lot Size")
ax.set_ylabel("% of Cities")

ax.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.show()

In [ ]:
plt.figure(figsize=(8, 6), dpi=200)

ax = sns.histplot(lots["lots_per_sq_km"], bins=25, kde=True, stat="percent")

sns.despine(ax=ax, top=True, right=True)

ax.set_title("Distribution of Lot Density")
ax.set_xlabel("Lots per Square Kilometer")
ax.set_ylabel("% of Cities")

ax.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.show()

In [ ]:
plt.figure(figsize=(8, 6), dpi=200)

ax = sns.histplot(lots["gini_coef"], bins=22, kde=True, stat="percent")

sns.despine(ax=ax, top=True, right=True)

ax.set_title("Distribution of Lot Gini Coefficients")
ax.set_xlabel("Gini Coefficient")
ax.set_ylabel("% of Cities")

ax.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.show()

In [ ]:
plt.figure(figsize=(8, 6), dpi=200)

ax = sns.histplot(lots["orientation_entropy"], bins=25, kde=True, stat="percent")

sns.despine(ax=ax, top=True, right=True)

ax.set_title("Distribution of Lot Orientation Entropies")
ax.set_xlabel("Orientation Entropy")
ax.set_ylabel("% of Cities")

ax.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.show()

In [ ]:
# Rose diagram for Houston
plot_rose_diagram(
    [get_orientation(p) for p in list(lots[lots["city"] == "houston-tx"].geometry.iloc[0].geoms)],
    "The rose diagram reveals the orientations for the parking lot shapes in Houston, Texas."
)

In [ ]:
# Rose diagram for San Antonio
plot_rose_diagram(
    [get_orientation(p) for p in list(lots[lots["city"] == "san-antonio-tx"].geometry.iloc[0].geoms)],
    "The lots San Antonio, Texas, are oriented in many different directions."
)

In [ ]:
# plot for lot size vs density
fig, ax = plt.subplots(figsize=(6,6), dpi=200)

ax = sns.regplot(    
    data=lots,
    x="avg_lot_area",
    y="lots_per_sq_km",
    scatter_kws={'alpha':0.7, 's':20},
    line_kws={'color':'black'},
    ax=ax
)

ax.set_title("Lot Size vs. Density")
ax.set_xlabel("Average Lot Size (km$^2$)")
ax.set_ylabel("Lots per Square Kilometer")

sns.despine(ax=ax, top=True, right=True)

plt.show()

In [ ]:
# pearson correlation between size and density
lots["avg_lot_area"].corr(lots["lots_per_sq_km"])

In [ ]:
# plot for lot size, density, and footprint
fig, ax = plt.subplots(figsize=(7,4), dpi=200)

ax = sns.scatterplot(    
    data=lots,
    x="avg_lot_area",
    y="lots_per_sq_km",
    hue="pct_lot_area",
    palette="viridis",
    alpha=0.7,
    s=20,
    ax=ax,
    legend=False
)

ax.set_title("Lot Size, Density, and Footprint")
ax.set_xlabel("Average Lot Size (km$^2$)")
ax.set_ylabel("Lots per Square Kilometer")

norm = plt.Normalize(lots['pct_lot_area'].min(), lots['pct_lot_area'].max())
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Lot Area %")

fig.text(
        0.5, -0.09, 
        "Average lot size * Density = Lot Area %. Cities may have the same footprint with high density and small lots or low density and big lots.",
        ha="center", 
        fontsize=10, 
        style='italic',
        wrap=True
    )

sns.despine(ax=ax, top=True, right=True)

plt.show()